In [1]:
# test sorting by pitch within polyphony

import numpy as np


rolls = np.array([
    [
        [0, 100, 2],
        [0, 10, 1],
        [0, 3, 1]
    ],
    [
        [0, 20, 2],
        [0, 50, 1],
        [0, 2, 1]
    ],
    [
        [0, 30, 2],
        [0, 71, 1],
        [0, 1, 1]
    ]
])
print(rolls.shape)
polyphony_counts = np.array([3, 3, 3])

for i in range(3):
    sorted_indices = np.argsort(rolls[i, :polyphony_counts[i], 1])
    rolls[i, :polyphony_counts[i]] = rolls[i, :polyphony_counts[i]][sorted_indices]

rolls

(3, 3, 3)


array([[[  0,   3,   1],
        [  0,  10,   1],
        [  0, 100,   2]],

       [[  0,   2,   1],
        [  0,  20,   2],
        [  0,  50,   1]],

       [[  0,   1,   1],
        [  0,  30,   2],
        [  0,  71,   1]]])

In [9]:
import torch

torch.tensor(rolls.reshape(3, -1)).unsqueeze(0)

tensor([[[  0,   3,   1,   0,  10,   1,   0, 100,   2],
         [  0,   2,   1,   0,  20,   2,   0,  50,   1],
         [  0,   1,   1,   0,  30,   2,   0,  71,   1]]])

In [3]:
rolls.reshape(3, -1)

array([[  0,   3,   1,   0,  10,   1,   0, 100,   2],
       [  0,   2,   1,   0,  20,   2,   0,  50,   1],
       [  0,   1,   1,   0,  30,   2,   0,  71,   1]])

In [4]:
a = {'a': 1, 'b': 2, 'c': 3}
b = {**a, 'a': 'bowen'}
b

{'a': 'bowen', 'b': 2, 'c': 3}

In [26]:
DURATION_TEMPLATES = np.array([1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768, 1024, 1536, 2048, 3072, 4096])

def _notes_to_rolls(notes: list, max_tick: int, max_polyphony=4, program=0):
    """
    Converts a list of musical notes into a tensor representation (a "piano roll").

    This function creates a tensor where each row represents a tick and each column
    contains information about the notes starting at that tick.

    Args:
        notes (list): A list of note dictionaries, each with 'pitch', 'tick', 'duration'.
                        The ticks in this list are expected to be relative to the start
                        of the desired tensor, not absolute performance ticks.
        max_tick (int): The total number of ticks for the resulting tensor (its length).
        max_polyphony (int): The maximum number of simultaneous notes allowed per tick.
        program (int): The instrument program number (0 for melody, 1 for accompaniment).

    Returns:
        torch.Tensor: A tensor of shape `(max_tick, max_polyphony * 3)` ready for
                        the model's `preprocess` step. Each note is represented by
                        3 numbers: (program, pitch, duration_idx). The tensor is
                        padded with 255 for empty slots.
    """
    # DURATION_TEMPLATES contains the quantized durations the model understands.
    # We find the midpoints between these templates to decide which template a
    # given note duration is closest to.
    duration_boundaries = (DURATION_TEMPLATES[1:] + DURATION_TEMPLATES[:-1]) / 2.0
    
    # Initialize the piano roll tensor.
    # Shape: (time, polyphony, features)
    # Features are (instrument_program, pitch, duration_index).
    # We use 255 as a special padding value, which is later ignored by the model.
    rolls = np.full((max_tick, max_polyphony, 3), dtype=np.uint8, fill_value=255)
    
    # Keep track of how many notes are at each tick to handle polyphony.
    polyphony_counts = np.zeros(max_tick, dtype=np.uint8)

    for note in notes:
        # TEEHEE CHECKPOINT | SHOULD BE OK
        tick = note['tick']
        if tick >= max_tick:
            continue

        # Drop notes that exceed the maximum polyphony for a given tick.
        if polyphony_counts[tick] >= max_polyphony:
            print(f"Warning: Exceeded max polyphony at tick {tick}. Note with pitch {note['pitch']} dropped.")
            continue
        
        # Find the index of the closest duration template.
        duration_idx = np.searchsorted(duration_boundaries, note['duration'])

        # Place the note's data into the correct slot in the tensor.
        slot = polyphony_counts[tick]
        rolls[tick, slot, 0] = program
        rolls[tick, slot, 1] = note['pitch']
        rolls[tick, slot, 2] = duration_idx
        
        polyphony_counts[tick] += 1

    # For polyphonic ticks, sort the notes by pitch. This creates a canonical
    # representation, which helps the model learn more effectively.
    for i in range(max_tick):
        if polyphony_counts[i] > 1:
            sorted_indices = np.argsort(rolls[i, :polyphony_counts[i], 1])
            rolls[i, :polyphony_counts[i]] = rolls[i, :polyphony_counts[i]][sorted_indices]

    # Reshape the tensor to the final format expected by the model's preprocess function.
    # Shape becomes (max_tick, max_polyphony * 3), e.g., (48, 12).
    return torch.tensor(rolls.reshape(max_tick, -1))

In [27]:
import json
melody_history = json.load(open('test_melody_history.json'))['melody_notes']
melody_history
# # Note: We do NOT update the accompaniment history until after generation.

# # Step 2: Create a prompt for the model from history occurring BEFORE the generation start tick.
# # This logic ensures the model always gets a fixed-size input and that the most
# # recent history is right-aligned (padded at the front).
generation_start_tick = 100
prompt_length_ticks = 50
prompt_end_tick = generation_start_tick
print(f'prompt_end_tick: {prompt_end_tick}')
prompt_start_tick = max(0, prompt_end_tick - prompt_length_ticks)
print(f'prompt_start_tick: {prompt_start_tick}')

# This is the fixed context length the model expects.
model_context_len = prompt_length_ticks
# This is the actual duration of the history we have available for the prompt.
actual_history_duration = prompt_end_tick - prompt_start_tick

# # If there's no history to use, there's nothing to generate from.
# if actual_history_duration <= 0:
#     print ([], preprocess_start_time, time.perf_counter(), time.perf_counter(), time.perf_counter())

# # Calculate the padding needed at the beginning of the context window.
padding_duration = model_context_len - actual_history_duration

# # Filter notes and make their ticks relative to the *padded* window.
# # This aligns the existing history to the END of the context window.
# # For example, if padding_duration is 10, the first note's tick will be 10, not 0.
prompt_melody = [
    {**n, 'tick': (n['tick'] - prompt_start_tick) + padding_duration}
    for n in melody_history if prompt_start_tick <= n['tick'] < prompt_end_tick
]
prompt_melody
# prompt_acc = [
#     {**n, 'tick': (n['tick'] - prompt_start_tick) + padding_duration}
#     for n in accompaniment_history if prompt_start_tick <= n['tick'] < prompt_end_tick
# ]

prompt_end_tick: 100
prompt_start_tick: 50


[{'pitch': 77, 'tick': 1, 'duration': 0},
 {'pitch': 68, 'tick': 1, 'duration': 1},
 {'pitch': 67, 'tick': 1, 'duration': 2},
 {'pitch': 65, 'tick': 1, 'duration': 3},
 {'pitch': 71, 'tick': 3, 'duration': 0},
 {'pitch': 60, 'tick': 3, 'duration': 1},
 {'pitch': 78, 'tick': 3, 'duration': 2},
 {'pitch': 66, 'tick': 4, 'duration': 0},
 {'pitch': 67, 'tick': 5, 'duration': 0},
 {'pitch': 79, 'tick': 5, 'duration': 1},
 {'pitch': 66, 'tick': 5, 'duration': 2},
 {'pitch': 77, 'tick': 5, 'duration': 3},
 {'pitch': 77, 'tick': 6, 'duration': 0},
 {'pitch': 62, 'tick': 6, 'duration': 1},
 {'pitch': 78, 'tick': 7, 'duration': 0},
 {'pitch': 66, 'tick': 7, 'duration': 1},
 {'pitch': 67, 'tick': 7, 'duration': 2},
 {'pitch': 71, 'tick': 7, 'duration': 3},
 {'pitch': 72, 'tick': 8, 'duration': 0},
 {'pitch': 66, 'tick': 8, 'duration': 1},
 {'pitch': 74, 'tick': 8, 'duration': 2},
 {'pitch': 73, 'tick': 9, 'duration': 0},
 {'pitch': 71, 'tick': 10, 'duration': 0},
 {'pitch': 72, 'tick': 10, 'durat

In [29]:
x_mel_raw = _notes_to_rolls(prompt_melody, model_context_len, 4, program=0)
x_mel_raw

tensor([[255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255],
        [  0,  65,   2,   0,  67,   1,   0,  68,   0,   0,  77,   0],
        [255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255],
        [  0,  60,   0,   0,  71,   0,   0,  78,   1, 255, 255, 255],
        [  0,  66,   0, 255, 255, 255, 255, 255, 255, 255, 255, 255],
        [  0,  66,   1,   0,  67,   0,   0,  77,   2,   0,  79,   0],
        [  0,  62,   0,   0,  77,   0, 255, 255, 255, 255, 255, 255],
        [  0,  66,   0,   0,  67,   1,   0,  71,   2,   0,  78,   0],
        [  0,  66,   0,   0,  72,   0,   0,  74,   1, 255, 255, 255],
        [  0,  73,   0, 255, 255, 255, 255, 255, 255, 255, 255, 255],
        [  0,  68,   1,   0,  71,   0,   0,  72,   0, 255, 255, 255],
        [  0,  62,   0, 255, 255, 255, 255, 255, 255, 255, 255, 255],
        [255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255],
        [  0,  66,   0, 255, 255, 255, 255, 255, 255, 255, 255, 255],
        [  0,  60,  

In [30]:
x_mel_raw = x_mel_raw.unsqueeze(0)
x_mel_raw

tensor([[[255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255],
         [  0,  65,   2,   0,  67,   1,   0,  68,   0,   0,  77,   0],
         [255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255],
         [  0,  60,   0,   0,  71,   0,   0,  78,   1, 255, 255, 255],
         [  0,  66,   0, 255, 255, 255, 255, 255, 255, 255, 255, 255],
         [  0,  66,   1,   0,  67,   0,   0,  77,   2,   0,  79,   0],
         [  0,  62,   0,   0,  77,   0, 255, 255, 255, 255, 255, 255],
         [  0,  66,   0,   0,  67,   1,   0,  71,   2,   0,  78,   0],
         [  0,  66,   0,   0,  72,   0,   0,  74,   1, 255, 255, 255],
         [  0,  73,   0, 255, 255, 255, 255, 255, 255, 255, 255, 255],
         [  0,  68,   1,   0,  71,   0,   0,  72,   0, 255, 255, 255],
         [  0,  62,   0, 255, 255, 255, 255, 255, 255, 255, 255, 255],
         [255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255, 255],
         [  0,  66,   0, 255, 255, 255, 255, 255, 255, 255, 255, 255],
      

In [31]:
x_mel_raw.shape

torch.Size([1, 50, 12])

In [15]:
l = []
for i in range(100):
    for j in range(np.random.randint(0, 5)):
        l.append({'pitch': np.random.randint(60, 80), 'tick': i, 'duration': j})
l

[{'pitch': 76, 'tick': 0, 'duration': 0},
 {'pitch': 72, 'tick': 0, 'duration': 1},
 {'pitch': 72, 'tick': 0, 'duration': 2},
 {'pitch': 78, 'tick': 0, 'duration': 3},
 {'pitch': 76, 'tick': 1, 'duration': 0},
 {'pitch': 77, 'tick': 1, 'duration': 1},
 {'pitch': 66, 'tick': 1, 'duration': 2},
 {'pitch': 74, 'tick': 1, 'duration': 3},
 {'pitch': 65, 'tick': 2, 'duration': 0},
 {'pitch': 61, 'tick': 3, 'duration': 0},
 {'pitch': 71, 'tick': 3, 'duration': 1},
 {'pitch': 65, 'tick': 5, 'duration': 0},
 {'pitch': 69, 'tick': 5, 'duration': 1},
 {'pitch': 78, 'tick': 7, 'duration': 0},
 {'pitch': 69, 'tick': 7, 'duration': 1},
 {'pitch': 62, 'tick': 7, 'duration': 2},
 {'pitch': 72, 'tick': 8, 'duration': 0},
 {'pitch': 62, 'tick': 8, 'duration': 1},
 {'pitch': 74, 'tick': 8, 'duration': 2},
 {'pitch': 71, 'tick': 9, 'duration': 0},
 {'pitch': 68, 'tick': 9, 'duration': 1},
 {'pitch': 78, 'tick': 9, 'duration': 2},
 {'pitch': 79, 'tick': 11, 'duration': 0},
 {'pitch': 74, 'tick': 11, 'durat